# Import Libraries

In [58]:
import pandas as pd
import numpy as np
import collections

# Import DataFrame

In [59]:
sampled_df_filepath = "sampled_dataframe.pkl"

sampled_df = pd.read_pickle(sampled_df_filepath).reset_index(drop=True)

### Getting Unique Vocabulary

In [60]:
vocab = set()

for sublist in sampled_df['lemmas']:
    vocab.update(sublist)

vocab_size = len(vocab)

print(f"Vocab size is {vocab_size} lemmas.")

Vocab size is 159862 lemmas.


### Generating BoW Representations

<div class="alert alert-block alert-danger">
  <b>Danger:</b> Do not run the following on the entire dataset.
</div>

In [66]:
token_frequencies = collections.Counter()

for sentence in sampled_df["lemmas"]:
    token_frequencies.update(sentence)

sorted_counts = np.sort(np.fromiter(token_frequencies.values(), dtype=np.uint32), descending=True)
cumulative_sorted_counts = np.cumsum(sorted_counts)
normalized_cumulative_sorted_counts = cumulative_sorted_counts / cumulative_sorted_counts[-1]

FREQUENCY_THRESHOLD = 0.8
threshold = np.searchsorted(normalized_cumulative_sorted_counts, FREQUENCY_THRESHOLD) + 1

print(f"We will use the most frequent {threshold} lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.")

We will use the most frequent 1126 lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.


In [67]:
top_lemmas_with_count = token_frequencies.most_common(threshold)
vocab = [lemma for lemma, _ in top_lemmas_with_count]

vocab_size = len(vocab) + 1  # 1 for <UNK> (vocab_size - 1)
print(f"Vocab size is {vocab_size} lemmas.")

# 1. Build the mapping dictionary
word_to_id = {word: idx for idx, word in enumerate(vocab)}
word_to_id['<UNK>'] = vocab_size - 1

# 2. Build BoW array
bow_representations = np.zeros((sampled_df.shape[0], vocab_size), dtype=np.uint16)
unk_id = word_to_id['<UNK>']

for i, sentence in enumerate(sampled_df['lemmas']):
    for lemma in sentence:
        bow_representations[i, word_to_id.get(lemma, unk_id)] += 1

Vocab size is 1127 lemmas.


In [68]:
bow_representations

array([[ 0,  0,  1, ...,  0,  0,  1],
       [ 4,  3,  1, ...,  0,  0,  5],
       [ 3,  1,  0, ...,  0,  0,  4],
       ...,
       [ 2,  2,  2, ...,  0,  0,  0],
       [ 3,  3,  2, ...,  0,  0,  7],
       [ 4,  5,  3, ...,  0,  0, 19]], shape=(509960, 1127), dtype=uint16)

# 1D Convolutions

The following cell determines the minimum vocabulary size needed to cover 80% of all token occurrences in the corpus by calculating cumulative relative word frequencies.

In [38]:
leftover_counts = cumulative_sorted_counts[-1] - cumulative_sorted_counts[threshold - 1]
leftover_counts

np.uint64(2749079)

Creating dictionary from vocab

In [40]:
word_to_id = {word: idx for idx, word in enumerate(vocab, start=1)}
word_to_id['<PAD>'] = 0
word_to_id['<UNK>'] = vocab_size - 1

Generating int representations

In [41]:
max_length = 50
num_samples = sampled_df.shape[0]
int_representations = np.zeros((num_samples, max_length), dtype=np.uint16)
unk_id = word_to_id['<UNK>']

for (i, sentence) in enumerate(sampled_df['lemmas']):
    sentence = sentence[:max_length]

    for (j, lemma) in enumerate(sentence):
        int_representations[i, j] = word_to_id.get(lemma, unk_id)

In [ ]:
bow_representations = np.eye()

In [42]:
print(num_samples)

509960


In [ ]:
sampled_df.columns
sampled_df

Index(['tokens', 'lemmas', 'join_key', 'length', 'Date', 'Speaker_role',
       'Speaker_MP', 'Speaker_minister', 'Speaker_party', 'Party_orientation',
       'Speaker_ID', 'Speaker_name'],
      dtype='str')

In [54]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()

target_column = 'Party_orientation'
y = le.fit_transform(sampled_df[target_column])
X = int_representations

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

Defining Model Architecture

In [57]:
import torch.nn as nn

class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        output_channels = 64
        output_features = 10
        K = 3
        self.conv1 = nn.Conv1d(vocab_size, output_channels, K)
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.fc1 = nn.Linear(output_channels, output_features)

ModuleNotFoundError: No module named 'torch'